# 03 — Trading Environment Validation
This notebook validates `src.environment` behavior for Gymnasium compliance, legal action handling, reward consistency, and state-shape integrity.

Checks include deterministic resets, `reset/step/render` behavior, mask legality, invalid-action handling, and rollout assertions.

In [1]:
from __future__ import annotations

from pathlib import Path
import random
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from src.utils.config_loader import resolve_config
from src.utils.seed import set_global_seed
from src.workflows.data_workflow import run_data_workflow
from src.environment.registry import make_env
from src.reward.reward_factory import build_reward_engine

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'configs').exists():
            return candidate
    raise FileNotFoundError('Repository root not found.')

ROOT = find_repo_root(Path.cwd())
CONFIG = resolve_config(root=str(ROOT))
SEED = int(CONFIG.get('training', {}).get('random_seed', 42))
set_global_seed(SEED, deterministic_torch=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PAIR = CONFIG['data']['pairs'][0]
data = run_data_workflow(CONFIG, pairs=[PAIR], root=str(ROOT))
train_df = data[PAIR]['train']

reward_engine = build_reward_engine(CONFIG)
env = make_env(train_df, PAIR, CONFIG, reward_engine=reward_engine)
print(f'PAIR={PAIR}, train_rows={len(train_df):,}, action_space_n={env.action_space.n}')

2026-03-13 03:17:38 | src.utils.seed | INFO | Global seed set to 42 (deterministic_torch=True)


PAIR=EURUSD, train_rows=7,951, action_space_n=10


In [2]:
obs_a, info_a = env.reset(seed=SEED)
obs_b, info_b = env.reset(seed=SEED)

# Reset(seed=...) determinism check
assert np.allclose(obs_a['flat'], obs_b['flat']), 'reset(seed) is not deterministic for identical seed'

# Observation/API schema checks
assert set(obs_a.keys()) == {'market', 'portfolio', 'mask', 'flat'}
assert obs_a['market'].ndim == 2, 'market observation must be [window, features]'
assert obs_a['portfolio'].ndim == 1, 'portfolio observation must be vector'
assert obs_a['mask'].shape[0] == env.action_space.n, 'mask/action size mismatch'
assert obs_a['flat'].ndim == 1, 'flat observation must be 1D'
assert np.isfinite(obs_a['flat']).all(), 'flat observation contains non-finite values'

display(pd.DataFrame([
    {
        'window_length': obs_a['market'].shape[0],
        'num_features': obs_a['market'].shape[1],
        'portfolio_dim': obs_a['portfolio'].shape[0],
        'num_actions': obs_a['mask'].shape[0],
        'flat_dim': obs_a['flat'].shape[0],
    }
]))

,window_length,num_features,portfolio_dim,num_actions,flat_dim
0,24,19,10,10,476


## Step-level API, reward, and invalid-action behavior
We verify legal action execution, reward breakdown consistency, and safe handling of out-of-bounds actions.

In [3]:
obs, info = env.reset(seed=SEED + 1)
legal_actions = np.where(obs['mask'] > 0)[0]
assert len(legal_actions) > 0, 'No legal action available at reset state'

legal_action = int(legal_actions[0])
next_obs, reward, terminated, truncated, step_info = env.step(legal_action)

assert isinstance(reward, float), 'Reward must be float'
assert isinstance(terminated, bool) and isinstance(truncated, bool), 'terminated/truncated must be bool'
assert set(next_obs.keys()) == {'market', 'portfolio', 'mask', 'flat'}
assert step_info.get('was_legal') is True, 'Chosen legal action was flagged illegal'
assert 0 <= step_info.get('executed_action', 0) < env.action_space.n, 'Executed action out of bounds'

reward_breakdown = step_info.get('reward_breakdown', {})
if reward_breakdown:
    assert 'total_normalized' in reward_breakdown
    assert np.isfinite(reward_breakdown['total_normalized'])
    assert abs(reward - float(reward_breakdown['total_normalized'])) < 1e-6

# Invalid action index should be handled by policy (not crash)
obs2, _ = env.reset(seed=SEED + 2)
_, reward_bad, _, _, bad_info = env.step(env.action_space.n + 999)
assert bad_info.get('was_legal') is False, 'Out-of-range action should be illegal'
assert bad_info.get('action_name') == 'HOLD', 'Invalid action should degrade to HOLD under current policy'

display(pd.DataFrame([
    {
        'legal_action_used': legal_action,
        'reward': reward,
        'invalid_action_reward': reward_bad,
        'invalid_action_policy': bad_info.get('invalid_action_policy'),
        'terminated': terminated,
        'truncated': truncated,
    }
]))

,legal_action_used,reward,invalid_action_reward,invalid_action_policy,terminated,truncated
0,0,0.0,0.0,convert_to_hold_with_penalty,False,False


## Action legality and state integrity across rollout
We run a bounded rollout using legal random actions and assert finite state/reward values at every step.

In [4]:
obs, info = env.reset(seed=SEED + 3)
rollout_steps = 300
records = []

for t in range(rollout_steps):
    legal = np.where(obs['mask'] > 0)[0]
    assert len(legal) > 0, f'No legal action at step {t}'
    action = int(np.random.choice(legal))
    obs, reward, terminated, truncated, info = env.step(action)

    # Integrity checks
    assert np.isfinite(obs['flat']).all(), f'Non-finite flat observation at step {t}'
    assert np.isfinite(reward), f'Non-finite reward at step {t}'
    assert info.get('mask') is not None, f'Missing legality mask in info at step {t}'
    assert obs['mask'].shape[0] == env.action_space.n, f'Mask/action mismatch at step {t}'

    records.append({
        't': t,
        'action': action,
        'was_legal': bool(info.get('was_legal', False)),
        'equity': float(info.get('equity', np.nan)),
        'drawdown': float(info.get('drawdown', np.nan)),
        'forced_liquidation': bool(info.get('forced_liquidation', False)),
    })

    if terminated or truncated:
        break

rollout_df = pd.DataFrame(records)
display(rollout_df.head())
display(rollout_df.tail())
print(f'Completed rollout steps: {len(rollout_df)}')

,t,action,was_legal,equity,drawdown,forced_liquidation
0,0,2,True,100002.15,0.000000,False
1,1,9,True,100000.75,0.000014,False
2,2,0,True,100005.85,0.000000,False
3,3,8,True,100004.65,0.000012,False
4,4,2,True,99993.75,0.000121,False


,t,action,was_legal,equity,drawdown,forced_liquidation
295,295,0,True,99431.959589,0.005739,False
296,296,1,True,99451.059589,0.005548,False
297,297,0,True,99439.559589,0.005663,False
298,298,9,True,99429.359589,0.005765,False
299,299,7,True,99427.470548,0.005783,False


Completed rollout steps: 300


## Environment assumptions and API notes
- Gymnasium-style seeding is validated via `reset(seed=...)` (there is no separate `seed()` method in this implementation).
- `render()` is checked for callable behavior and graceful handling of missing render modes.
- Invalid actions are expected to degrade to HOLD under configured policy with optional penalty.

In [5]:
render_status = 'not-called'
try:
    _render_out = env.render()
    render_status = f'ok ({type(_render_out).__name__})'
except NotImplementedError:
    render_status = 'not-implemented (acceptable for headless mode)'
except Exception as exc:
    render_status = f'error: {exc}'

final_checks = pd.DataFrame([
    {
        'seed_reset_deterministic': True,
        'step_api_validated': True,
        'invalid_action_policy_validated': True,
        'rollout_integrity_validated': True,
        'render_status': render_status,
        'rollout_steps_checked': int(len(rollout_df)),
    }
])
display(final_checks)
print('✅ Environment validation checks completed.')

,seed_reset_deterministic,step_api_validated,invalid_action_policy_validated,rollout_integrity_validated,render_status,rollout_steps_checked
0,True,True,True,True,not-implemented (acceptable for headless mode),300


✅ Environment validation checks completed.
